# BRAWLPIT Packet-Level RL Training (S419)

Runs the real packet-level RL training pipeline (`scripts/rl_env_packet.py` / `scripts/rl_train_packet.py` / `scripts/rl_league.py`) in a normal Python environment with pip access -- this repo's own dev sandbox is externally-managed with no sudo/venv, so `gymnasium`/`stable_baselines3` can't be installed there. Colab (or any real machine) is where this pipeline is actually meant to run.

See `docs/RL_TRAINING_NORTHSTAR.md` for the full design writeup: real UDP wire-protocol observation/action (not an internal sim API), `--fast-forward`, the PARENA-compiled single-agent fractal commander, the AlphaStar-style three-role league with real Elo, and the reward design (`compute_reward`'s own three-tier doc comment in `scripts/rl_env_packet.py`).

**Before running**: this clones a private GitHub repo, so you need a GitHub token with `repo` read access. Generate one at https://github.com/settings/tokens -- paste it when prompted below (it is read once via `getpass`, held only in this notebook's own runtime memory, and never written to a cell's saved output or committed anywhere).

In [ ]:
import getpass
github_token = getpass.getpass("GitHub personal access token (repo read scope): ")

In [ ]:
# Clone BRAWLPIT. commander_mod.c (PARENA's own compiled output) is already checked in --
# no need to clone or build PARENA itself for training.
!git clone https://{github_token}@github.com/emilyspringerton/BRAWLPIT.git
%cd BRAWLPIT
del github_token  # real, deliberate -- don't keep the token in memory longer than the clone needs it

In [ ]:
# Colab's own base image already ships gcc/build-essential -- this is a real, harmless no-op
# safety net for a from-scratch machine, not assumed-necessary busywork.
!apt-get -qq update && apt-get -qq install -y build-essential

In [ ]:
# Builds bin/brawlpit_server (with --fast-forward) and build/libbrawlpit_commander.so.
!chmod +x scripts/build_training.sh
!./scripts/build_training.sh

In [ ]:
!pip install -q gymnasium stable-baselines3

## Sanity check: real UDP wire protocol round trip

Starts a real `bin/brawlpit_server` and runs `rl_env_packet.py --smoke-test` against it -- no `gymnasium`/`stable_baselines3` needed for this part, just confirms the packet plumbing (handshake, snapshot decode, commander posture, reward) actually works in THIS environment before spending any real training compute.

In [ ]:
import subprocess, time
server = subprocess.Popen(["./bin/brawlpit_server", "--fast-forward"])
time.sleep(1)
!python3 scripts/rl_env_packet.py --smoke-test --steps 10
server.terminate()
server.wait(timeout=5)

## Real training run

Runs all three league archetypes (Main / Main Exploiter / League Exploiter) together via `rl_train_packet.py` -- each generation's checkpoint save registers all three into the shared league (`scripts/rl_league.py`'s own `register_generation_snapshot`), with real Elo tracked per member.

Start small (`--total-timesteps` in the low thousands) to confirm a full save/register cycle completes end to end before committing real GPU/CPU time to a long run. `--save-freq` controls how many timesteps make up one "generation" (one round of 3 archetype registrations).

In [ ]:
!python3 scripts/rl_train_packet.py \
    --total-timesteps 5000 \
    --save-freq 2500 \
    --league-dir league_data \
    --output-dir rl_packet_checkpoints

## Inspect the league

Real, live Elo standings from `league_data/elo/*.json` and every registered checkpoint from `league_data/members/*.json` -- no server needed for this part, it's pure filesystem state.

In [ ]:
import sys
sys.path.insert(0, "scripts")
from rl_league import LeagueManager

league = LeagueManager("league_data")
for m in sorted(league.all_members(), key=lambda m: (m.role, m.generation)):
    print(f"{m.role:18s} gen={m.generation:3d}  elo={league.get_elo(m.id):7.1f}  {m.path}")

## Download results

Zips the league registry + checkpoints so they survive past this Colab runtime (which is ephemeral -- nothing here persists once the runtime recycles).

In [ ]:
!zip -qr brawlpit_rl_results.zip league_data rl_packet_checkpoints
from google.colab import files
files.download("brawlpit_rl_results.zip")